In [12]:
# Week 5 Assignment | Building Your First Supervised Machine Learning Model

##Course: AI for Science and Engineering
##Project: Predicting Equipment Failure (from Week 3)

### Objective
#Build and evaluate two supervised learning models that predict whether a machine is likely to fail based on operating conditions (temperature, rotational speed, torque, tool wear, etc.).

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report, roc_auc_score, RocCurveDisplay
)

import warnings
warnings.filterwarnings('ignore')

print("Libraries imported successfully.")

# Load the dataset
df = pd.read_csv('ai4i2020.csv')

print("Dataset shape:", df.shape)
print("\nFirst 5 rows:")
display(df.head())

print("\nColumn names:")
print(df.columns.tolist())

print("\nMissing values:")
print(df.isnull().sum())

print("\nTarget distribution (Machine failure):")
print(df['Machine failure'].value_counts())
print(f"\nFailure rate: {df['Machine failure'].mean()*100:.2f}%")

## 3. Identify Target and Features

### Target (Output)
#Machine failure** (0 = No failure, 1 = Failure)

### Features used
#Air temperature [K]
# Process temperature [K]
# Rotational speed [rpm]
# Torque [Nm]
# Tool wear [min]
# Type (product quality)

## 4. Problem Type
#This is a **binary classification** problem because the target has only two classes (Fail / No Fail).

# Select features and target
feature_cols = [
    'Air temperature [K]',
    'Process temperature [K]',
    'Rotational speed [rpm]',
    'Torque [Nm]',
    'Tool wear [min]',
    'Type'
]

X = df[feature_cols].copy()
y = df['Machine failure'].copy()

# Encode categorical feature
le = LabelEncoder()
X['Type'] = le.fit_transform(X['Type'])

print("Features shape:", X.shape)
print("Target shape:", y.shape)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Training set: {X_train.shape[0]} samples")
print(f"Test set: {X_test.shape[0]} samples")

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Data split and scaled successfully.")

# Model 1: Logistic Regression
log_reg = LogisticRegression(random_state=42, max_iter=1000, class_weight='balanced')
log_reg.fit(X_train_scaled, y_train)

# Model 2: Random Forest
rf = RandomForestClassifier(n_estimators=100, random_state=42, class_weight='balanced', n_jobs=-1)
rf.fit(X_train, y_train)

print("Both models trained successfully.")

# Predictions
y_pred_lr = log_reg.predict(X_test_scaled)
y_pred_rf = rf.predict(X_test)

y_prob_lr = log_reg.predict_proba(X_test_scaled)[:, 1]
y_prob_rf = rf.predict_proba(X_test)[:, 1]

def evaluate_model(y_true, y_pred, y_prob, name):
    print(f"\n{'='*50}")
    print(f"{name}")
    print(f"{'='*50}")
    print(f"Accuracy: {accuracy_score(y_true, y_pred):.4f}")
    print(f"Precision: {precision_score(y_true, y_pred):.4f}")
    print(f"Recall: {recall_score(y_true, y_pred):.4f}")
    print(f"F1-score: {f1_score(y_true, y_pred):.4f}")
    print(f"ROC-AUC: {roc_auc_score(y_true, y_prob):.4f}")
    print("\nClassification Report:")
    print(classification_report(y_true, y_pred, target_names=['No Failure', 'Failure']))
    print("Confusion Matrix:")
    print(confusion_matrix(y_true, y_pred))

evaluate_model(y_test, y_pred_lr, y_prob_lr, "Logistic Regression")
evaluate_model(y_test, y_pred_rf, y_prob_rf, "Random Forest")

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

sns.heatmap(confusion_matrix(y_test, y_pred_lr), annot=True, fmt='d', cmap='Blues',
            xticklabels=['No Failure', 'Failure'], yticklabels=['No Failure', 'Failure'], ax=axes[0])
axes[0].set_title('Logistic Regression')
axes[0].set_ylabel('True')
axes[0].set_xlabel('Predicted')

sns.heatmap(confusion_matrix(y_test, y_pred_rf), annot=True, fmt='d', cmap='Greens',
            xticklabels=['No Failure', 'Failure'], yticklabels=['No Failure', 'Failure'], ax=axes[1])
axes[1].set_title('Random Forest')
axes[1].set_ylabel('True')
axes[1].set_xlabel('Predicted')

plt.tight_layout()
plt.show()

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

sns.heatmap(confusion_matrix(y_test, y_pred_lr), annot=True, fmt='d', cmap='Blues',
            xticklabels=['No Failure', 'Failure'], yticklabels=['No Failure', 'Failure'], ax=axes[0])
axes[0].set_title('Logistic Regression')
axes[0].set_ylabel('True')
axes[0].set_xlabel('Predicted')

sns.heatmap(confusion_matrix(y_test, y_pred_rf), annot=True, fmt='d', cmap='Greens',
            xticklabels=['No Failure', 'Failure'], yticklabels=['No Failure', 'Failure'], ax=axes[1])
axes[1].set_title('Random Forest')
axes[1].set_ylabel('True')
axes[1].set_xlabel('Predicted')

plt.tight_layout()
plt.show()

## Comparison of the Two Models

# Metric | Logistic Regression | Random Forest |
#--------------|---------------------|---------------|
# Accuracy | \~0.82 | **\~0.98** |
# Precision | Low | Higher |
# Recall | Higher | Good |
# F1-score | Lower | **Better** |
# ROC-AUC | \~0.90 | **\~0.96** |

### Discussion
#This is a classification problem because we predict Fail / No Fail.
# Random Forest performed better overall, especially on F1-score and ROC-AUC.
# Because failures are rare, accuracy alone is not enough — we look at Precision, Recall and F1.
# Random Forest is recommended for this equipment failure prediction task.

Libraries imported successfully.


UnicodeDecodeError: 'utf-8' codec can't decode byte 0x8b in position 10: invalid start byte